# Bronze to Silver

This notebook follows one direct batch flow:

**Bronze files → `parse_aemo()` → structured views → candidate Delta tables → Silver Delta tables → validation**

Monthly, Daily and Current are merged into one actual-demand table. AEMO forecasts remain separate because each target time appears in several forecast runs.

This is deliberately a **batch baseline**: every run reads all matching Bronze CSV files and recreates all four Delta tables. That is easy to understand and correct for exploration, but it repeats work when only a few new files arrive. A later notebook will retain the same parsing and deduplication rules while changing how new files are discovered and stored.

In [ ]:
from pyspark.sql import functions as F

## Parse the AEMO files

AEMO CSVs contain several record groups, not one ordinary header and table. `spark.read.text()` preserves each line so the parser can find the exact `I` row that defines a group and retain only its matching `D` rows.

The row pattern must be explicit: `DISPATCH,REGIONSUM` contains actual demand, while `P5MIN,REGIONSOLUTION` contains AEMO forecasts.

Spark DataFrames are **lazy**. Operations such as `filter()`, `select()` and `withColumn()` define a logical processing plan; they do not immediately load and store the complete dataset in memory. The `.first()` call is the first action: Spark reads enough data to obtain the `I` row. The larger `D`-row plan runs later when SQL creates the Delta tables.

In [ ]:
def parse_aemo(
    path,
    row_pattern,
    time_column,
    forecast_time_column=None,
    filter_runno=False,
):
    # Batch read: every matching CSV file is part of this DataFrame plan.
    raw = spark.read.text(path)

    # The I row supplies the column positions for this AEMO section.
    # first() is an action, so Spark executes this small header lookup now.
    header_row = (
        raw
        .filter(F.col("value").startswith(f"I,{row_pattern},"))
        .select(F.split("value", ",").alias("fields"))
        .first()
    )

    if header_row is None:
        raise ValueError(f"No I row found for {row_pattern} in {path}")

    header = header_row["fields"]
    time_i = header.index(time_column)
    region_i = header.index("REGIONID")
    intervention_i = header.index("INTERVENTION")
    demand_i = header.index("TOTALDEMAND")

    # Keep the matching D rows and the operational VIC1 demand record.
    # These transformations remain lazy until a later action needs the data.
    rows = (
        raw
        .filter(F.col("value").startswith(f"D,{row_pattern},"))
        .withColumn("fields", F.split("value", ","))
        .filter(F.col("fields")[region_i] == "VIC1")
        .filter(F.col("fields")[intervention_i].cast("int") == 0)
    )

    if filter_runno:
        runno_i = header.index("RUNNO")
        rows = rows.filter(F.col("fields")[runno_i].cast("int") == 1)

    time = F.to_timestamp(
        F.regexp_replace(F.col("fields")[time_i], '"', ""),
        "yyyy/MM/dd HH:mm:ss",
    ).alias("time")

    demand = (
        F.regexp_replace(F.col("fields")[demand_i], '"', "")
        .cast("double")
        .alias("demand")
    )

    # Actual demand needs one time; forecasts need creation and target times.
    if forecast_time_column is None:
        return rows.select(time, demand)

    forecast_time_i = header.index(forecast_time_column)
    forecast_time = F.to_timestamp(
        F.regexp_replace(F.col("fields")[forecast_time_i], '"', ""),
        "yyyy/MM/dd HH:mm:ss",
    ).alias("forecast_time")

    return rows.select(forecast_time, time, demand)

## Actual demand

Monthly, Daily and Current contain the same observed VIC1 demand delivered through different AEMO layers. Each call defines a DataFrame with `time | demand` and applies `RUNNO = 1`. No complete result is stored yet.

The sources overlap. The project assumes fresher publications may contain corrections, so priority is Current, then Daily, then Monthly.

In [ ]:
bronze_path = "/Volumes/workspace/default/aemo_mlops_volume/bronze"

monthly = (
    parse_aemo(
        f"{bronze_path}/monthly_uncompressed/*.CSV",
        row_pattern="DISPATCH,REGIONSUM",
        time_column="SETTLEMENTDATE",
        filter_runno=True,
    )
    .withColumn("priority", F.lit(1))
)

daily = (
    parse_aemo(
        f"{bronze_path}/daily_uncompressed/*.CSV",
        row_pattern="DISPATCH,REGIONSUM",
        time_column="SETTLEMENTDATE",
        filter_runno=True,
    )
    .withColumn("priority", F.lit(2))
)

current = (
    parse_aemo(
        f"{bronze_path}/current_uncompressed/*.CSV",
        row_pattern="DISPATCH,REGIONSUM",
        time_column="SETTLEMENTDATE",
        filter_runno=True,
    )
    .withColumn("priority", F.lit(3))
)

## AEMO demand forecast

P5MIN produces a new forecast run every five minutes. `forecast_time` is when AEMO produced the run; `time` is the future interval being predicted. The pair `(forecast_time, time)` identifies one forecast observation.

Archive and Current are kept as separate DataFrames. The batch notebook still reads every file from both folders, but this separation will later allow each source to track new files independently.

In [ ]:
forecast_archive = parse_aemo(
    f"{bronze_path}/forecast/archive_uncompressed/*.CSV",
    row_pattern="P5MIN,REGIONSOLUTION",
    time_column="INTERVAL_DATETIME",
    forecast_time_column="RUN_DATETIME",
)

forecast_current = parse_aemo(
    f"{bronze_path}/forecast/current_uncompressed/*.CSV",
    row_pattern="P5MIN,REGIONSOLUTION",
    time_column="INTERVAL_DATETIME",
    forecast_time_column="RUN_DATETIME",
)

## Create the structured views

A temporary view gives SQL a name for a DataFrame plan. It does not copy the data or create a permanent table. The views last only for this Spark session.

When a later SQL statement queries a view, Spark executes its parsing plan against the Bronze files. From this point onward, SQL can work with regular columns rather than AEMO's line-based format.

In [ ]:
monthly.createOrReplaceTempView("monthly")
daily.createOrReplaceTempView("daily")
current.createOrReplaceTempView("current")
forecast_archive.createOrReplaceTempView("forecast_archive")
forecast_current.createOrReplaceTempView("forecast_current")

## Build the actual candidate table

`UNION ALL` combines the three sources without hiding their overlap. The candidate table persists those rows in Delta together with source priority, making the later selection rule explicit.

`CREATE OR REPLACE` triggers the lazy DataFrame plans and rewrites the complete table. Therefore this remains a batch process: adding one new Bronze file still causes every matching CSV to be read again.

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.default.demand_vic_5min_candidates
USING DELTA
AS

SELECT time, demand, priority FROM monthly
UNION ALL
SELECT time, demand, priority FROM daily
UNION ALL
SELECT time, demand, priority FROM current

## Build the actual Silver table

The final table keeps one demand value per timestamp. `ROW_NUMBER()` ranks overlapping candidates by priority, so Current wins over Daily and Daily wins over Monthly.

This second `CREATE OR REPLACE` also rebuilds the full Silver table on every run.

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.default.demand_vic_5min
USING DELTA
AS

WITH ranked AS (
    SELECT
        time,
        demand,
        ROW_NUMBER() OVER (
            PARTITION BY time
            ORDER BY priority DESC
        ) AS row_number
    FROM workspace.default.demand_vic_5min_candidates
)

SELECT time, demand
FROM ranked
WHERE row_number = 1

## Build the forecast candidate table

Archive and Current are combined with `UNION ALL`, so exact overlaps remain visible in the candidate table. As with actual demand, `CREATE OR REPLACE` reads both complete folders and rewrites the full Delta table.

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.default.aemo_demand_forecast_vic_5min_candidates
USING DELTA
AS

SELECT forecast_time, time, demand FROM forecast_archive
UNION ALL
SELECT forecast_time, time, demand FROM forecast_current

## Build the forecast Silver table

Archive and Current forecast files can overlap. `DISTINCT` removes only identical rows and preserves every distinct forecast value. Both times remain necessary: `forecast_time` says when the prediction was created, while `time` says when its predicted demand should occur.

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.default.aemo_demand_forecast_vic_5min
USING DELTA
AS

SELECT DISTINCT
    forecast_time,
    time,
    demand
FROM workspace.default.aemo_demand_forecast_vic_5min_candidates

## Validate the Silver outputs

The first check confirms that both final tables contain rows, cover a plausible time range and have no missing keys or demand values. The second check verifies their required uniqueness. A healthy run returns two summary rows with `invalid_rows = 0`, followed by an empty duplicate-key result.

In [ ]:
%sql
SELECT
    'actual demand' AS dataset,
    COUNT(*) AS rows,
    MIN(time) AS first_target_time,
    MAX(time) AS last_target_time,
    COUNT_IF(time IS NULL OR demand IS NULL) AS invalid_rows
FROM workspace.default.demand_vic_5min

UNION ALL

SELECT
    'AEMO forecast' AS dataset,
    COUNT(*) AS rows,
    MIN(time) AS first_target_time,
    MAX(time) AS last_target_time,
    COUNT_IF(forecast_time IS NULL OR time IS NULL OR demand IS NULL) AS invalid_rows
FROM workspace.default.aemo_demand_forecast_vic_5min

In [ ]:
%sql
SELECT 'actual demand' AS dataset, time AS first_key, CAST(NULL AS TIMESTAMP) AS second_key, COUNT(*) AS rows
FROM workspace.default.demand_vic_5min
GROUP BY time
HAVING COUNT(*) > 1

UNION ALL

SELECT 'AEMO forecast' AS dataset, forecast_time AS first_key, time AS second_key, COUNT(*) AS rows
FROM workspace.default.aemo_demand_forecast_vic_5min
GROUP BY forecast_time, time
HAVING COUNT(*) > 1

## Validate conflicting values

Exact duplicates are harmless, but different demand values for the same logical key deserve inspection. The first query exposes changes between Monthly, Daily and Current before source priority is applied. Those differences may represent fresher corrections. The second checks whether one AEMO forecast key still has more than one demand value after exact deduplication.

In [ ]:
%sql
SELECT
    time,
    SORT_ARRAY(COLLECT_SET(demand)) AS conflicting_values
FROM workspace.default.demand_vic_5min_candidates
GROUP BY time
HAVING COUNT(DISTINCT demand) > 1
ORDER BY time

In [ ]:
%sql
SELECT
    forecast_time,
    time,
    SORT_ARRAY(COLLECT_SET(demand)) AS conflicting_values
FROM workspace.default.aemo_demand_forecast_vic_5min
GROUP BY forecast_time, time
HAVING COUNT(DISTINCT demand) > 1
ORDER BY forecast_time, time

## What this batch baseline establishes

The notebook now has one reusable AEMO parser, explicit actual and forecast inputs, visible overlap handling, persistent Delta outputs and small validation checks. Its limitation is equally explicit: every run scans all Bronze files and recreates every table.

The later incremental version should change file discovery and table maintenance—not the parsing contract, source priority, output schemas or validation meaning.